In [2]:
# Instalação de Dependências e Importações (Código)
%pip install -q psycopg2-binary fastavro boto3

import io
import psycopg2
import boto3
from fastavro import writer, parse_schema

# Configuracoes de Infraestrutura (Rede interna do Docker)
config_db = {"host": "postgres", "database": "postgres", "user": "postgres", "password": "senha123", "port": 5432}
config_s3 = {"endpoint_url": "http://minio:9000", "aws_access_key_id": "root-minio", "aws_secret_access_key": "root12345678", "region_name": "local"}

bucket_name = "raw"
destino_s3 = "python_native/ledger_avro/ledger_stream.avro"

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Definição Rígida do Contrato de Dados (Código)
 
print("Declarando o esquema JSON para governanca do arquivo Avro...")

avro_schema = {
    "name": "TransacaoFinanceira",
    "namespace": "mba.datacollect",
    "type": "record",
    "fields": [
        {"name": "id_transacao", "type": "string"},
        {"name": "conta_origem", "type": "string"},
        {"name": "conta_destino", "type": "string"},
        {"name": "valor", "type": "string"},
        {"name": "moeda", "type": "string"},
        {"name": "data_transacao", "type": "string"}
    ]
}
parsed_schema = parse_schema(avro_schema)
print("Esquema validado com sucesso pela biblioteca fastavro.")

Declarando o esquema JSON para governanca do arquivo Avro...
Esquema validado com sucesso pela biblioteca fastavro.


In [4]:
# Conexão e Extração via Cursor Nativo (Código)

print("Conectando ao PostgreSQL e extraindo dados para serializacao...")

# Inicializamos a conexao fora do bloco 'with' para permitir que ela seja fechada na ultima celula
conn = psycopg2.connect(**config_db)

with conn.cursor() as cursor:
    cursor.execute("SELECT id_transacao, conta_origem, conta_destino, valor, moeda, data_transacao FROM transacoes_financeiras;")
    
    registros = []
    for linha in cursor.fetchall():
        registros.append({
            "id_transacao": str(linha[0]), 
            "conta_origem": linha[1], 
            "conta_destino": linha[2],
            "valor": str(linha[3]), 
            "moeda": linha[4], 
            "data_transacao": str(linha[5])
        })

print(f"Mapeamento concluido para {len(registros)} dicionarios estruturados.")

Conectando ao PostgreSQL e extraindo dados para serializacao...
Mapeamento concluido para 20000 dicionarios estruturados.


In [5]:
# Serialização Binária e Carga para o MinIO (Código)

print("Incorporando Schema JSON e serializando dados em binario orientado a linhas...")
buffer_binario = io.BytesIO()
writer(buffer_binario, parsed_schema, registros)

print(f"Fazendo upload do pacote Avro para s3://{bucket_name}/{destino_s3}...")
s3_client = boto3.client('s3', **config_s3)
s3_client.put_object(
    Bucket=bucket_name,
    Key=destino_s3,
    Body=buffer_binario.getvalue(),
    ContentType='application/octet-stream'
)

print("Ingestao Avro para o MinIO concluida com sucesso!")

Incorporando Schema JSON e serializando dados em binario orientado a linhas...
Fazendo upload do pacote Avro para s3://raw/python_native/ledger_avro/ledger_stream.avro...
Ingestao Avro para o MinIO concluida com sucesso!


In [6]:
# Encerramento de Conexoes

print("Fechando a conexao com o banco de dados PostgreSQL...")
conn.close()
print("Conexao encerrada com seguranca. Os recursos do banco foram liberados.")

Fechando a conexao com o banco de dados PostgreSQL...
Conexao encerrada com seguranca. Os recursos do banco foram liberados.


### Analise de Arquitetura: O Formato Avro no Python Nativo

**Caracteristicas Principais:** Formato binario, autodescritivo e estritamente orientado a linhas (Row-based).

#### Vantagens
* **Governanca por Contrato:** Os dados nao sao persistidos se violarem a estrutura declarada no esquema JSON. O esquema viaja embutido no cabecalho do arquivo.
* **Escrita Sequencial Ultrarapida:** Por focar em linhas, a insercao e o append incremental de novos blocos nao exigem reorganizacao de indices complexos.
* **Evolucao de Esquema Segura:** Suporta adicoes ou remocoes de colunas no tempo mantendo retrocompatibilidade.

#### Desvantagens
* **Ineficiente para Queries Analiticas Agregadas:** Forca a varredura completa da linha (I/O inutilizado) caso queira analisar apenas uma coluna especifica.

#### Caso de Uso no Mercado
* Formato padrao para pipelines de mensageria em tempo real (como barramentos Apache Kafka) e Landing Zones que exijam alta taxa de transferencia de escrita.

In [1]:
# Instalação de Dependências e Importações (Código)
%pip install -q psycopg2-binary fastavro boto3

import io
import psycopg2
import boto3
from fastavro import writer, parse_schema

# Configuracoes de Infraestrutura (Rede interna do Docker)
config_db = {"host": "postgres", "database": "postgres", "user": "postgres", "password": "senha123", "port": 5432}
config_s3 = {"endpoint_url": "http://minio:9000", "aws_access_key_id": "root-minio", "aws_secret_access_key": "root12345678", "region_name": "local"}

bucket_name = "raw"
destino_s3 = "python_native/ledger_avro/ledger_stream.avro"

Note: you may need to restart the kernel to use updated packages.
